In [11]:
import pandas as pd
import numpy as np
from DATA.stock_invest_function import *
from pykrx import stock

In [32]:
def get_kospi_krx(start="20000101", end=None):
    """
    KRX(한국거래소)에서 코스피 지수(코드: 1001)의 일별 OHLCV를 가져옵니다.
    날짜 형식: 'YYYYMMDD'
    """
    if end is None:
        end = pd.Timestamp.today().strftime("%Y%m%d")
    df = stock.get_index_ohlcv_by_date(start, end, "1001")  # 1001 = KOSPI
    df = df.rename(columns={"시가":"open","고가":"high","저가":"low","종가":"close","거래량":"volume","거래대금":"value"})
    df.index.name = "date"
    df = df.reset_index()
    # pykrx는 이미 한국 시간/영업일 기준
    return df[["date","open","high","low","close","volume","value"]]

In [33]:
# DB 접속 정보 설정
db_info = {
    'user': 'stox7412',         # 예: 'root'
    'password': 'Apt106503!~', # 예: '1234'
    # 'host': '192.168.0.230',
    'host': get_db_host(),         # 예: 'localhost' 또는 IP
    'port': '3307',              # 기본 포트는 보통 3306
    'database': 'investar'        # 예: 'trade_data'
}

#### RIM Valuation 을 위한 데이터 수집.. 개벌 기업 베타 추정

In [46]:
ticker = '043150'
prc_raw = fetch_table_data(db_info, "KSE_Price")

stock_price = prc_raw[prc_raw['code'] == ticker]
daily_price = stock_price[['date', 'close']]
daily_price.rename(columns={'close': 'equity_close'}, inplace=True)

✅ 'KSE_Price' 테이블에서 6280830건의 데이터를 가져왔습니다.


In [47]:
# 사용 예시
kospi_krx = get_kospi_krx(start="20100101")
kospi_index = kospi_krx[['date', 'close']]
kospi_index.rename(columns={'close': 'index_close'}, inplace=True)

# date 컬럼을 datetime으로 통일
kospi_index["date"] = pd.to_datetime(kospi_index["date"])
daily_price["date"] = pd.to_datetime(daily_price["date"])

# inner join
price_df = pd.merge(kospi_index, daily_price, on="date", how="inner")

In [48]:
from datetime import datetime, timedelta
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# 데이터 전처리
df = price_df.copy()

df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

print("📈 KOSPI vs 개별기업 베타 분석")
print(f"분석 대상 기업 주가 범위: {df['equity_close'].min():,}원 ~ {df['equity_close'].max():,}원")
print(f"KOSPI 지수 범위: {df['index_close'].min():.2f} ~ {df['index_close'].max():.2f}")

# 수익률 계산 (일일 수익률)
df['index_return'] = df['index_close'].pct_change()
df['equity_return'] = df['equity_close'].pct_change()

# NaN 값 제거
df = df.dropna()

if len(df) == 0:
    print("오류: 유효한 데이터가 없습니다")
    exit()

# 전체 데이터 기간 계산
total_days = (df['date'].max() - df['date'].min()).days
total_years = total_days / 365.25

print(f"전체 데이터 기간: {total_years:.2f}년 ({len(df)}일)")

# 결과 저장용 딕셔너리
results = {}

# 계산할 기간들 (년)
periods = [5, 3, 1]

print("\n" + "="*60)
print("베타 지수 계산 결과")
print("="*60)

# 각 기간별 베타 계산
for period in periods:

    # 기간 조건 확인
    if total_years < 1:
        results[f'{period}년'] = "기간미달"
        print(f"\n📊 {period}년 베타:")
        print(f"   결과: 기간미달")
        continue
    elif period == 1 and total_years >= 1:
        pass  # 1년 베타 계산 가능
    elif period == 3 and total_years >= 3:
        pass  # 3년 베타 계산 가능
    elif period == 5 and total_years >= 5:
        pass  # 5년 베타 계산 가능
    else:
        continue  # 해당 기간 데이터 부족으로 건너뜀

    # 해당 기간의 데이터 추출
    start_date = df['date'].max() - timedelta(days=int(period * 365.25))
    period_data = df[df['date'] >= start_date].copy()

    if len(period_data) < 30:  # 최소 30일 데이터 필요
        results[f'{period}년'] = "데이터부족"
        print(f"\n📊 {period}년 베타:")
        print(f"   결과: 데이터부족")
        continue

    # 베타 계산 (선형회귀의 기울기)
    try:
        slope, intercept, r_value, p_value, std_err = stats.linregress(
            period_data['index_return'],
            period_data['equity_return']
        )

        beta = slope
        r_squared = r_value ** 2

        results[f'{period}년'] = {
            'beta': round(beta, 4),
            'r_squared': round(r_squared, 4),
            'p_value': round(p_value, 4),
            'data_points': len(period_data),
            'period_start': period_data['date'].min().strftime('%Y-%m-%d'),
            'period_end': period_data['date'].max().strftime('%Y-%m-%d')
        }

        # 결과 출력
        print(f"\n📊 {period}년 베타:")
        print(f"   베타 값: {beta:.4f}")
        print(f"   결정계수 (R²): {r_squared:.4f}")
        print(f"   유의확률 (p-value): {p_value:.4f}")
        print(f"   데이터 포인트: {len(period_data)}일")
        print(f"   분석기간: {period_data['date'].min().strftime('%Y-%m-%d')} ~ {period_data['date'].max().strftime('%Y-%m-%d')}")

        # 베타 해석
        if beta > 1.2:
            interpretation = "고베타 (시장보다 변동성 높음)"
        elif beta > 0.8:
            interpretation = "중베타 (시장과 유사한 변동성)"
        elif beta > 0:
            interpretation = "저베타 (시장보다 변동성 낮음)"
        else:
            interpretation = "음베타 (시장과 반대 방향)"

        print(f"   해석: {interpretation}")

    except Exception as e:
        results[f'{period}년'] = f"계산오류: {str(e)}"
        print(f"\n📊 {period}년 베타:")
        print(f"   결과: 계산오류 - {str(e)}")

# 최종 요약
print(f"\n" + "="*60)
print("베타 분석 요약")
print("="*60)

for period, result in results.items():
    if isinstance(result, dict):
        print(f"{period}: {result['beta']} (R² = {result['r_squared']})")
    else:
        print(f"{period}: {result}")

print(f"\n💡 베타 해석 가이드:")
print(f"   • 베타 > 1: 시장보다 변동성이 큰 공격적 주식")
print(f"   • 베타 = 1: 시장과 동일한 움직임")
print(f"   • 0 < 베타 < 1: 시장보다 안정적인 방어적 주식")
print(f"   • 베타 < 0: 시장과 반대 방향으로 움직임")
print(f"   • R² 값이 높을수록 베타의 신뢰도가 높음")

# ticker, period, beta 형태의 결과 DataFrame 생성
print(f"\n" + "="*60)
print("ticker, period, beta 형태의 결과 데이터")
print("="*60)

# 결과 데이터를 리스트로 정리
result_data = []

# 티커는 샘플이므로 'SAMPLE'로 설정 (실제 사용시 실제 티커로 변경)
# ticker = ticker   # 실제 사용시: ticker = 'YOUR_TICKER_CODE'

for period, result in results.items():
    if isinstance(result, dict):
        result_data.append({
            'ticker': ticker,
            'period': period,
            'beta': result['beta'],
            'r_squared': result['r_squared'],
            'p_value': result['p_value'],
            'data_points': result['data_points']
        })
    else:
        result_data.append({
            'ticker': ticker,
            'period': period,
            'beta': result,
            'r_squared': None,
            'p_value': None,
            'data_points': None
        })

# DataFrame으로 변환
beta_df = pd.DataFrame(result_data)

# 기본 3개 컬럼만 출력
beta_simple = beta_df[['ticker', 'period', 'beta']].copy()

print("📋 기본 결과 (ticker, period, beta):")
print(beta_simple.to_string(index=False))

print(f"\n📋 상세 결과 (추가 통계 포함):")
print(beta_df.to_string(index=False))

# CSV 파일로 저장 (선택사항)
# beta_simple.to_csv('beta_results_simple.csv', index=False, encoding='utf-8-sig')
# beta_df.to_csv('beta_results_detailed.csv', index=False, encoding='utf-8-sig')
# print(f"\n💾 결과가 CSV 파일로 저장되었습니다.")

print(f"\n📊 DataFrame 변수명:")
print(f"   • beta_simple: ticker, period, beta 컬럼만")
print(f"   • beta_df: 모든 통계 정보 포함")


📈 KOSPI vs 개별기업 베타 분석
분석 대상 기업 주가 범위: 8,800원 ~ 50,300원
KOSPI 지수 범위: 1457.64 ~ 3305.21
전체 데이터 기간: 12.19년 (2996일)

베타 지수 계산 결과

📊 5년 베타:
   베타 값: 0.8610
   결정계수 (R²): 0.1888
   유의확률 (p-value): 0.0000
   데이터 포인트: 1227일
   분석기간: 2020-08-03 ~ 2025-08-01
   해석: 중베타 (시장과 유사한 변동성)

📊 3년 베타:
   베타 값: 0.7592
   결정계수 (R²): 0.1784
   유의확률 (p-value): 0.0000
   데이터 포인트: 734일
   분석기간: 2022-08-02 ~ 2025-08-01
   해석: 저베타 (시장보다 변동성 낮음)

📊 1년 베타:
   베타 값: 0.7822
   결정계수 (R²): 0.3649
   유의확률 (p-value): 0.0000
   데이터 포인트: 242일
   분석기간: 2024-08-01 ~ 2025-08-01
   해석: 저베타 (시장보다 변동성 낮음)

베타 분석 요약
5년: 0.861 (R² = 0.1888)
3년: 0.7592 (R² = 0.1784)
1년: 0.7822 (R² = 0.3649)

💡 베타 해석 가이드:
   • 베타 > 1: 시장보다 변동성이 큰 공격적 주식
   • 베타 = 1: 시장과 동일한 움직임
   • 0 < 베타 < 1: 시장보다 안정적인 방어적 주식
   • 베타 < 0: 시장과 반대 방향으로 움직임
   • R² 값이 높을수록 베타의 신뢰도가 높음

ticker, period, beta 형태의 결과 데이터
📋 기본 결과 (ticker, period, beta):
ticker period   beta
043150     5년 0.8610
043150     3년 0.7592
043150     1년 0.7822

📋 상세 결과 (추가 통계 포함):
ticker period 

#### 채권금리 가져오기

In [50]:
economy_raw = fetch_table_data(db_info, "Korea_Economy_Data")

✅ 'Korea_Economy_Data' 테이블에서 9505건의 데이터를 가져왔습니다.


In [52]:
economy_raw['Index_Name'].unique().tolist()

['M2광의통화',
 '은행대출금_가계',
 '예금은행_수신금리(신규)',
 '예금은행_수신금리(잔액)',
 '예금은행_대출금리(신규)',
 '전산업생산지수',
 '제조업재고지수',
 '제조업가동률지수',
 '도소매업지수',
 '자동차판매액지수_계절조정',
 '설비투자지수',
 '설비용기계류생산지수',
 '국내수요기계수주액',
 '건설기성액_불변_계절조정',
 '건축허가면적',
 '건설수주액',
 '건축착공면적',
 '선행지수순환변동치',
 '소비자심리지수',
 'BSI',
 '경제심리지수',
 '취업자수',
 '경상수지',
 '수출금액지수',
 '수입금액지수',
 '소득교역조건지수',
 '외환보유액',
 '소비자물가지수',
 '생활물가지수',
 '생산자물가지수',
 '수출물가지수',
 '수입물가지수',
 '주택매매가격지수',
 '주택전세가격지수',
 '지가변동률',
 '미분양주택',
 '원면']